In [1]:
# JUSTIN

In [2]:
# GPU check
import torch, subprocess, textwrap
print("CUDA available:", torch.cuda.is_available())
!nvidia-smi

CUDA available: True
Mon Sep 22 19:14:53 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   55C    P8             10W /   70W |       2MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+--------------------------

In [3]:
# Clone the repo and set paths
REPO_URL = "https://github.com/MadKeyboardArtist/5703-Federated-Model.git"
REPO_DIR = "/content/5703-Federated-Model"

import os, sys
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
os.chdir(REPO_DIR)
sys.path.append(REPO_DIR)   # so Python can import your modules
print("CWD:", os.getcwd())

Cloning into '5703-Federated-Model'...
remote: Enumerating objects: 3737, done.
remote: Counting objects: 100% (19/19), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 3737 (delta 8), reused 12 (delta 5), pack-reused 3718 (from 1)
Receiving objects: 100% (3737/3737), 259.47 MiB | 37.31 MiB/s, done.
Resolving deltas: 100% (83/83), done.
Updating files: 100% (3697/3697), done.
CWD: /content/5703-Federated-Model


In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [5]:
from multihead_model import MultiHeadModel
from config import D_TABULAR, D_EMBEDDING, D_FUSION

In [6]:
# Configure modality and model
# Select which modality to train: "tabular" or "image"
MODALITY = "image"   # change to "image" if you want image training
NUM_CLASSES = 5        # Diabetes_012 is binary classification (0/1)

In [7]:
import importlib, multihead_model
importlib.reload(multihead_model)
from multihead_model import MultiHeadModel

In [8]:

model = MultiHeadModel(
    d_tabular = D_TABULAR,
    d_embedding = D_EMBEDDING,
    d_fusion = D_FUSION,
    n_tabular_classes = None,
    n_image_classes = NUM_CLASSES,
    n_multi_classes = None
    )

device = "cuda" if torch.cuda.is_available() else "cpu"
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=0.0)
model = model.to(device)


In [9]:
# Build DataLoaders
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import datasets, transforms
import pandas as pd
from pathlib import Path
import math
import torch

In [10]:
# ----- Image dataset (returns dict for consistency) -----
class ImageFolderDict(datasets.ImageFolder):
    def __getitem__(self, index):
        img, label = super().__getitem__(index)
        return {"img": img, "label": torch.tensor(label, dtype=torch.long)}

In [11]:
import image_basic_preprocessing
importlib.reload(image_basic_preprocessing)
from image_basic_preprocessing import tfms

IMG_TFMS = tfms
TRAIN_DIR = Path("image_dataset/split/train")
VAL_DIR   = Path("image_dataset/split/val")

In [12]:
def build_loaders(modality, batch_size=64, workers=4):
    if VAL_DIR.exists():
        train_ds = ImageFolderDict(TRAIN_DIR, transform=IMG_TFMS)
        val_ds   = ImageFolderDict(VAL_DIR,   transform=IMG_TFMS)
    else:
        full = ImageFolderDict(TRAIN_DIR, transform=IMG_TFMS)
        n = len(full); n_val = math.floor(0.2 * n)
        train_ds, val_ds = random_split(full, [n - n_val, n_val], generator=torch.Generator().manual_seed(42))
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=workers, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=workers, pin_memory=True)
    return train_loader, val_loader

train_loader, val_loader = build_loaders(MODALITY, batch_size=64, workers=4)

# quick check
b = next(iter(train_loader))
print("Batch keys:", list(b.keys()))
for k,v in b.items():
    if isinstance(v, torch.Tensor):
        print(k, tuple(v.shape), v.dtype)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Batch keys: ['img', 'label']
img (64, 3, 224, 224) torch.float32
label (64,) torch.int64


In [13]:
# Training and Evaluation loops
import torch.nn.functional as F
import torch

In [14]:
def train_one_epoch(model, loader, optimizer, device, modality, log_every=50):
    model.train()
    total, correct, loss_sum = 0, 0, 0.0

    for i, batch in enumerate(loader, 1):
        for k, v in batch.items():
            if isinstance(v, torch.Tensor):
                batch[k] = v.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        logits = model(task_type="image", x_img=batch["img"])
        loss = F.cross_entropy(logits, batch["label"])
        loss.backward()
        optimizer.step()

        bs = batch["label"].size(0)
        loss_sum += loss.item() * bs
        total    += bs
        correct  += (logits.argmax(1) == batch["label"]).sum().item()

        if i % log_every == 0:
            print(f"  step {i:4d} | loss {loss_sum/max(total,1):.4f} | acc {correct/max(total,1):.4f}")

    return {"loss": loss_sum/max(total,1), "acc": correct/max(total,1)}

@torch.no_grad()
def evaluate(model, loader, device, modality):
    model.eval()
    total, correct, loss_sum = 0, 0, 0.0

    for batch in loader:
        for k, v in batch.items():
            if isinstance(v, torch.Tensor):
                batch[k] = v.to(device, non_blocking=True)

        logits = model(task_type="image", x_img=batch["img"])
        loss = F.cross_entropy(logits, batch["label"])
        bs = batch["label"].size(0)
        loss_sum += loss.item() * bs
        total    += bs
        correct  += (logits.argmax(1) == batch["label"]).sum().item()

    return {"loss": loss_sum/max(total,1), "acc": correct/max(total,1)}

In [15]:
# Main training loop
import os, copy

def run_training(model, train_loader, val_loader, modality, epochs=10, save_dir="runs/exp_single"):
    os.makedirs(save_dir, exist_ok=True)
    best_ckpt = os.path.join(save_dir, "best.pth")

    best_val = float("inf")
    best_state = None

    for epoch in range(1, epochs+1):
        print(f"\nEpoch {epoch}/{epochs}")
        tr = train_one_epoch(model, train_loader, optimizer, device, modality, log_every=50)
        va = evaluate(model, val_loader, device, modality)

        print(f"train: loss={tr['loss']:.4f}, acc={tr['acc']:.4f} | "
              f"val: loss={va['loss']:.4f}, acc={va['acc']:.4f}")

        if va["loss"] < best_val - 1e-6:
            best_val = va["loss"]
            best_state = {
                "epoch": epoch,
                "model": copy.deepcopy(model.state_dict()),
                "optimizer": optimizer.state_dict(),
                "val_loss": best_val,
                "modality": modality
            }
            torch.save(best_state, best_ckpt)
            print(f"[best updated] → {best_ckpt}")

# Example run (only one modality at a time)
run_training(model, train_loader, val_loader, MODALITY, epochs=5, save_dir=f"runs/{MODALITY}_exp")



Epoch 1/5
train: loss=1.1118, acc=0.5667 | val: loss=0.9235, acc=0.6456
[best updated] → runs/image_exp/best.pth

Epoch 2/5
train: loss=0.8789, acc=0.6746 | val: loss=0.8599, acc=0.6835
[best updated] → runs/image_exp/best.pth

Epoch 3/5
train: loss=0.8418, acc=0.6978 | val: loss=0.8327, acc=0.6835
[best updated] → runs/image_exp/best.pth

Epoch 4/5
train: loss=0.8254, acc=0.6944 | val: loss=0.8382, acc=0.6878

Epoch 5/5
train: loss=0.8168, acc=0.6975 | val: loss=0.7975, acc=0.7004
[best updated] → runs/image_exp/best.pth
